In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np

# 1. Load Dataset
df = pd.read_csv("online_retail.csv", encoding="ISO-8859-1")

# 2. Clean Data
df = df.dropna(subset=['CustomerID', 'Description'])
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Date'] = df['InvoiceDate'].dt.date
df = df[df['Quantity'] > 0]

# Focus on Top 50 SKUs
top_skus = df.groupby('StockCode')['Quantity'].sum().nlargest(50).index
df_filtered = df[df['StockCode'].isin(top_skus)].copy()

# 3. Aggregate Daily Sales
daily_sales = df_filtered.groupby(['StockCode', 'Description', 'Date'])['Quantity'].sum().reset_index()

# 4. Supply Chain Metrics
summary_df = daily_sales.groupby(['StockCode', 'Description']).agg(
    Avg_Daily_Demand=('Quantity', 'mean'),
    Std_Daily_Demand=('Quantity', 'std')
).reset_index()

SERVICE_LEVEL_Z = 1.65  # 95% service level target
LEAD_TIME_DAYS = 10     # Supplier lead time

summary_df['Safety_Stock'] = (
    SERVICE_LEVEL_Z * summary_df['Std_Daily_Demand'] * np.sqrt(LEAD_TIME_DAYS)
).round(0)

summary_df['Reorder_Point_ROP'] = (
    (summary_df['Avg_Daily_Demand'] * LEAD_TIME_DAYS) + summary_df['Safety_Stock']
).round(0)

summary_df['Target_Inventory_45D'] = (summary_df['Avg_Daily_Demand'] * 45).round(0)
summary_df['Suggested_PO_Qty'] = (summary_df['Target_Inventory_45D'] - summary_df['Safety_Stock']).round(0)

# Export output
summary_df.to_csv("kaggle_inventory_replenishment_plan.csv", index=False)
print("Execution Complete! Output file generated.")

Execution Complete! Output file generated.
